# Dask: detailed scenarios

Scale pandas across partitions with a lazy task graph. Needs **dask** (dataframe extra), in the lab dependencies. Self-contained: we build a pandas frame and partition it. See **dask_cheatsheet.md**.

In [ ]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
rng = np.random.default_rng(3)

n = 3000
pdf = pd.DataFrame({
    "ticker": rng.choice(["ACME", "GLOBEX", "INITECH"], n),
    "day": rng.integers(0, 180, n),
    "ret": rng.normal(0.0004, 0.012, n),
})
ddf = dd.from_pandas(pdf, npartitions=4)
print("partitions:", ddf.npartitions)
ddf.head()

## Lazy vs compute

A Dask operation builds a graph; nothing runs until compute().

In [ ]:
lazy_mean = ddf.groupby("ticker")["ret"].mean()
print("type before compute:", type(lazy_mean).__name__)
print(lazy_mean.compute().round(4))

## Aggregations in parallel

In [ ]:
summary = ddf.groupby("ticker")["ret"].agg(["mean", "std", "count"]).compute()
print(summary.round(4))

## map_partitions: apply a pandas function to each partition

In [ ]:
def add_zscore(part):
    part = part.copy()
    part["z"] = (part["ret"] - part["ret"].mean()) / part["ret"].std()
    return part

out = ddf.map_partitions(add_zscore)
print(out.compute().head(3).round(3))

## dask.array: chunked NumPy

In [ ]:
import dask.array as da
x = da.random.random((10000, 100), chunks=(2500, 100))
print("mean of column 0:", round(float(x.mean(axis=0)[0].compute()), 4))

## Recap

Partition a frame, the lazy compute model, parallel groupby, map_partitions, and chunked arrays. For real scale, start a distributed Client() for the dashboard.